# 00 — Colab Setup (Fact / Legal Thesis / Noise)

This notebook sets up the Google Colab environment: mounts Drive, installs
dependencies, and validates GPU, paths, and dataset. **Run this first.**

## 1. Mount Google Drive and locate the project

⚠️ **Upload the ENTIRE PROJECT to Drive**, não só a pasta `Data`. A pasta
no Drive (por padrão `/content/drive/MyDrive/Fato_Tese_Ruido`) precisa conter:

```
Fato_Tese_Ruido/
├── requirements.txt
├── src/ftr/...        ← o módulo 'ftr' (sem isto: No module named 'ftr')
├── configs/
├── scripts/
└── Data/dataset_FINAL.json
```

The cell below **locates the project folder automatically** (by searching for
`src/ftr`) and shows a checklist of what is present or missing.

In [1]:
import os, sys, glob

# Mount Drive (Colab only).
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    print('Not running on Colab — using local path.')


def _looks_like_project(p):
    return os.path.isdir(os.path.join(p, 'src', 'ftr'))


def _autodetect_root():
    candidates = ['/content/drive/MyDrive/Fato_Tese_Ruido']
    # search for any folder containing src/ftr inside MyDrive (2 levels)
    for base in ('/content/drive/MyDrive', '/content/drive/My Drive'):
        candidates += glob.glob(os.path.join(base, '**', 'src', 'ftr'), recursive=False)
        candidates += [os.path.dirname(os.path.dirname(p))
                       for p in glob.glob(os.path.join(base, '*', 'src', 'ftr'))]
    for c in candidates:
        if c and _looks_like_project(c):
            return c
    return candidates[0] if candidates else os.getcwd()


PROJECT_ROOT = _autodetect_root() if IN_COLAB else os.getcwd()
os.environ['FTR_PROJECT_ROOT'] = PROJECT_ROOT
if os.path.join(PROJECT_ROOT, 'src') not in sys.path:
    sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))
print('PROJECT_ROOT =', PROJECT_ROOT)

# Required items (code + data).

required = {'src/ftr': 'folder (ftr module)', 'configs': 'folder',
            'scripts': 'folder', 'Data/dataset_FINAL.json': 'dataset'}
# Optional items (Drive sometimes converts .txt/.xlsx to Google Docs/Sheets,
# removing the extension — no problem, deps are installed explicitly below).
optional = {'requirements.txt': 'file (optional)'}
print('\nProject folder checklist:')
missing = []
for rel, kind in required.items():
    ok = os.path.exists(os.path.join(PROJECT_ROOT, rel))
    print(f'  [{"OK" if ok else "MISSING"}] {rel}  ({kind})')
    if not ok:
        missing.append(rel)
for rel, kind in optional.items():
    ok = os.path.exists(os.path.join(PROJECT_ROOT, rel))
    print(f'  [{"OK" if ok else "--"}] {rel}  ({kind})')
if missing:
    raise SystemExit('\nMISSING required items above. Upload the ENTIRE project folder to Drive (not just Data/).')
print('\nOK — all required items present. (requirements.txt is optional.)')

Mounted at /content/drive
PROJECT_ROOT = /content/drive/MyDrive/Fato_Tese_Ruido

Checklist da pasta do projeto:
  [OK] src/ftr  (pasta (módulo ftr))
  [OK] configs  (pasta)
  [OK] scripts  (pasta)
  [OK] Data/dataset_FINAL.json  (dataset)
  [--] requirements.txt  (arquivo (opcional))

OK — itens obrigatórios presentes. (requirements.txt é opcional.)


## 2. Install dependencies

Installs libraries explicitly (does not rely on `requirements.txt`, which Drive
sometimes converts to Google Docs format). torch is already available in Colab.

In [2]:
if IN_COLAB:
    !pip install -q 'transformers==4.48.3' 'datasets==3.3.0' 'scikit-learn==1.6.1' \
        evaluate sentencepiece sacremoses 'accelerate>=1.3.0' openpyxl seaborn pyarrow
    print('\nDependencies installed.')
else:
    print('Local: install with  pip install -r requirements.txt')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 78.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 484.9/484.9 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 65.2 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.12.0 which is incompatible.

Dependências instaladas.


## 3. Check GPU, environment and paths

In [3]:
import torch
from ftr.env import describe_environment, get_paths
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
import json
print(json.dumps(describe_environment(), indent=2, ensure_ascii=False))

CUDA disponível: False
{
  "in_colab": true,
  "drive_mounted": true,
  "project_root": "/content/drive/MyDrive/Fato_Tese_Ruido",
  "root_exists": true,
  "raw_dataset": "/content/drive/MyDrive/Fato_Tese_Ruido/Data/dataset_FINAL.json",
  "raw_dataset_exists": true,
  "python": "3.12.13",
  "platform": "linux"
}


## 4. Check that the dataset is in place

In [4]:
paths = get_paths()
print('Raw dataset:', paths.raw_dataset)
assert paths.raw_dataset.exists(), (
    f'Dataset not found at {paths.raw_dataset}. '
    'Upload Data/dataset_FINAL.json to Drive.')
print('OK — dataset found (%.1f MB).' % (paths.raw_dataset.stat().st_size/1e6))

Dataset bruto: /content/drive/MyDrive/Fato_Tese_Ruido/Data/dataset_FINAL.json
OK — dataset encontrado (259.0 MB).


## 5. Ready!

Environment ready. You can now:
- Rodar `00_EDA.ipynb` para a análise exploratória;
- Rodar `01_experiment_template.ipynb` para treinar um modelo;
- Rodar `02_run_all_colab.ipynb` para treinar todos em sequência.

Before training, prepare the data (cleaning + anonymisation + split):

In [5]:
# -W all shows all warnings (important to catch leakage alerts)
!cd "{PROJECT_ROOT}" && python -W all scripts/01_prepare_data.py --config configs/_base.yaml

2026-06-02 17:57:57.826522: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-06-02 17:57:57.940801: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
  prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
  parse = parser.parseString(pattern)
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:89: PyparsingDeprecationWarning: 'resetCache' depr